# Model Optimization Tutorial

This tutorial describe the process of optimizing the user's model. The input to this tutorial is a HAR file in Hailo Model state (before optimization; with native weights) and the output will be a quantized HAR file with quantized weights.

Note: For full information about Optimization and Quantization, refer to the `Dataflow Compiler user guide / Model optimization` section.

**Requirements:**

* Run this code in Jupyter notebook. See the Introduction tutorial for more details.
* The user should review the complete Parsing Tutorial (or created the HAR file in other way)

**Recommendation:**

* To obtain best performance run this code with a GPU machine. For full information see the `Dataflow Compiler user guide / Model optimization` section.

**Contents:**

* Quick optimization tutorial
* In-depth optimization & evaluation tutorial
* Advanced Model Modifications tutorial
* Compression and Optimization levels

In [3]:
# General imports used throughout the tutorial
# file operations
import json
import os

import numpy as np
import tensorflow as tf
from IPython.display import SVG
from matplotlib import patches
from matplotlib import pyplot as plt
from PIL import Image
from tensorflow.python.eager.context import eager_mode

# import the hailo sdk client relevant classes
from hailo_sdk_client import ClientRunner, InferenceContext

%matplotlib inline

IMAGES_TO_VISUALIZE = 5

## Quick Optimization Tutorial

After the HAR file has been created (using either `runner.translate_tf_model` or `runner.translate_onnx_model`), the next step is to go through the optimization process.

The basic optimization is performed just by calling `runner.optimize(calib_dataset)` (or the CLI `hailo optimize` command), as described on the user guide on: Building Models / Model optimization / Model Optimization Workflow.
The calibration dataset should be preprocessed according to the model's input requirements and it is recommended to have at least 1024 inputs and to use a GPU.
During this step it is also possible to use a model script which change the default behavior of the Dataflow Compiler, for example, to add additional layer for normalization.
All the model script available commands are described in the user guide on: Building Models / Model optimization / Optimization Related Model Script Commands.

In order to learn how to deal with common pitfalls, image formats and accuracy, refer to the in-depth section.

In [6]:
import torchvision as tv
import torch

def preproc(image, output_height=640, output_width=640):
    preprocess = tv.transforms.Compose([
        tv.transforms.Resize((output_height, output_width)),
    ])
    
    data = np.array(preprocess(image))
    
    return data

data_batch_size = 16
images_path = "../data/coco/images/val2017" # "../data"
images_list = [img_name for img_name in os.listdir(images_path) if os.path.splitext(img_name)[1] == ".jpg"]
calib_dataset = np.zeros((data_batch_size, 640, 640, 3))
for idx, img_name in enumerate(sorted(images_list)):
    if idx==data_batch_size:
        break
    img = Image.open(os.path.join(images_path, img_name)).convert('RGB')
    img_preproc = preproc(img)
    calib_dataset[idx, :, :, :] = img_preproc


In [288]:
calib_dataset.shape # should be (1500, 640, 640, 3)

(1024, 640, 640, 3)

In [16]:
# load our parsed HAR from the Parsing Tutorial

# model_name = "yolo11n"
# hailo_model_har_name = f"{model_name}_hailo_model.har"
# assert os.path.isfile(hailo_model_har_name), "Please provide valid path for HAR file"
# runner = ClientRunner(har=hailo_model_har_name)

# or load a previously quantized model
model_name = "yolo11n"
quantized_model_har_path = f"{model_name}_quantized_model_opl4.har"
assert os.path.isfile(quantized_model_har_path), "Please provide valid path for HAR file"
runner = ClientRunner(har=quantized_model_har_path)

In [ ]:
# Now we will create a model script, that tells the compiler to add a normalization on the beginning
# of the model (that is why we didn't normalize the calibration set;
# Otherwise we would have to normalize it before using it)

# Batch size is 8 by default
alls =  """
normalization1 = normalization([0.0, 0.0, 0.0], [255.0, 255.0, 255.0])
change_output_activation(conv54, sigmoid)
change_output_activation(conv65, sigmoid)
change_output_activation(conv80, sigmoid)
nms_postprocess("./yolo11n_nms_config.json", meta_arch=yolov8, engine=cpu)
allocator_param(width_splitter_defuse=disabled)
 """
#model_optimization_flavor(optimization_level=4, compression_level=4)


# Load the model script to ClientRunner so it will be considered on optimization
runner.load_model_script(alls)

# Call Optimize to perform the optimization process
runner.optimize(calib_dataset)

# Save the result state to a Quantized HAR file
quantized_model_har_path = f"{model_name}_quantized_model_opl4.har"
runner.save_har(quantized_model_har_path)

In [ ]:
# !hailomz eval yolov11n --har /local/workspace/hailo_virtualenv/lib/python3.10/site-packages/hailo_tutorials/notebooks/yolo11n_quantized_model_opl4.har --target=emulator
# !hailomz eval yolov11n --har /local/workspace/hailo_virtualenv/lib/python3.10/site-packages/hailo_tutorials/notebooks/yolo11n_quantized_model.har --target=emulator
# !hailomz eval yolov11n --har /local/workspace/hailo_virtualenv/lib/python3.10/site-packages/hailo_tutorials/notebooks/yolo11n_quantized_model.har

In [7]:
sample_dataset = np.zeros((2, 640, 640, 3))
SAMPLE_IMAGE_PATH = '../data/coco/images/val2017/000000000139.jpg'
img = Image.open(SAMPLE_IMAGE_PATH).convert('RGB')
img_preproc = preproc(img)
sample_dataset[0,:,:,:] = img_preproc

In [17]:
with runner.infer_context(InferenceContext.SDK_QUANTIZED) as ctx:
    output = runner.infer(ctx, sample_dataset[:1, :, :, :])

[info] Using 1 GPU for inference


Inference: 8entries [00:33,  4.20s/entries]


In [14]:
output.shape

(1, 80, 5, 100)

In [18]:
tv_dets = output[:, 62, :, :].reshape(5,100) # this is classid=62, which is a television
x = tv_dets.transpose()[:2, :4]
x640 = x * 640
x

# array([[0.39236316, 0.00957916, 0.6129946 , 0.24244972],
#        [0.48750573, 0.8681353 , 0.6807421 , 0.99895185]], dtype=float32)

array([[0.39236316, 0.00957916, 0.6129946 , 0.24244972],
       [0.48750573, 0.8681353 , 0.6807421 , 0.99895185]], dtype=float32)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
results = model.predict(sample_dataset[0,:,:,:], imgsz=640, conf=0.2)
# Process results list
for result in results:
    boxes = result.boxes  # Boxes object for bounding box outputs
    masks = result.masks  # Masks object for segmentation masks outputs
    keypoints = result.keypoints  # Keypoints object for pose outputs
    probs = result.probs  # Probs object for classification outputs
    obb = result.obb  # Oriented boxes object for OBB outputs
    #result.show()  # display to screen
    result.save(filename="result.jpg")  # save to disk
    
boxes[boxes.cls==62]

That concludes the quick tutorial.

In [293]:
# YOLO output (after multiplying by 640)
x1, y1, x2, y2 = 6.130665, 251.11243, 392.31656, 154.3751

# Original image size
W_orig = 640  # Replace with actual width
H_orig = 426   # Replace with actual height

# Scaling factors
scale_x = W_orig / 640
scale_y = H_orig / 640

# Rescale the coordinates
x1_orig = int(x1 * scale_x)
y1_orig = int(y1 * scale_y)
x2_orig = int(x2 * scale_x)
y2_orig = int(y2 * scale_y)

print(x1_orig, y1_orig, x2_orig, y2_orig)

6 167 392 102


In [ ]:
# YOLO output (after multiplying by 640)
x1, y1, x2, y2 = 6.130665, 251.11243, 392.31656, 154.3751

# Original image size
W_orig = 640  # Replace with actual width
H_orig = 426   # Replace with actual height

# Scaling factors
scale_x = W_orig / 640
scale_y = H_orig / 640

# Rescale the coordinates
x1_orig = int(x1 * scale_x)
y1_orig = int(y1 * scale_y)
x2_orig = int(x2 * scale_x)
y2_orig = int(y2 * scale_y)

print(x1_orig, y1_orig, x2_orig, y2_orig)

import cv2
import numpy as np

# Load the image
SAMPLE_IMAGE_PATH = '../data/coco/images/val2017/000000000139.jpg'
img = cv2.imread(SAMPLE_IMAGE_PATH)

# Get image dimensions
height, width, _ = img.shape

# Ground truth labels in YOLO format (class_id, x_center, y_center, bbox_width, bbox_height)
labels = [
    (62, 0.39044347, 0.01051836, 0.6155355, 0.24089317),
    (62, 0.127641, 0.505153, 0.233312, 0.2227),
    (62, 0.934195, 0.583462, 0.127109, 0.184812)
]
#0.39044347, 0.01051836, 0.6155355 , 0.24089317, 0.9146729 ],
      # [0.48778272, 0.86963475, 0.6793633 , 0.999899  , 0.5476935 ]],
      #dtype=float32)

# Loop through labels and draw bounding boxes
for class_id, x_center, y_center, bbox_width, bbox_height in labels:
    # Convert normalized YOLO coordinates to pixel values
    x_center, y_center = int(x_center * width), int(y_center * height)
    bbox_width, bbox_height = int(bbox_width * width), int(bbox_height * height)

    # Calculate top-left and bottom-right corners
    x1 = int(x_center - bbox_width / 2)
    y1 = int(y_center - bbox_height / 2)
    x2 = int(x_center + bbox_width / 2)
    y2 = int(y_center + bbox_height / 2)

    # Draw the bounding box
    color = (0, 255, 0)  # Green color for bounding box
    thickness = 2
    cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)

    # Add class label text
    cv2.putText(img, str(class_id), (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

# Display the image with bounding boxes
cv2.imshow("Ground Truth", img)
cv2.waitKey(0)  # Wait for key press
cv2.destroyAllWindows()  # Close window

In [ ]:
# Display the image with bounding boxes
cv2.imshow("Ground Truth", img)
cv2.waitKey(0)  # Wait for key press
cv2.destroyAllWindows()  # Close window

In [ ]:
!ls -la

In [ ]:
!hailomz eval yolov11n --har=/local/workspace/hailo_virtualenv/lib/python3.10/site-packages/hailo_tutorials/notebooks/yolov11n.har --data-path=/local/shared_with_docker/.hailomz/models_files/visdrone/2020-05-25/visdrone_val.tfrecord --target=emulator

In [ ]:
import tensorflow as tf

def is_tfrecord_corrupted(tfrecord_file):
    try:
        for record in tf.data.TFRecordDataset(tfrecord_file):
            # Attempt to parse the record
            _ = tf.train.Example.FromString(record.numpy())
    except tf.errors.DataLossError as e:
        print(f"DataLossError encountered: {e}")
        return True
    except Exception as e:
        print(f"An error occurred: {e}")
        return True
    return False

# Replace with your TFRecord file paths 
tfrecord_files = ['/local/workspace/hailo_virtualenv/lib/python3.10/site-packages/hailo_tutorials/data/visdrone_custom_val.tfrecord']

for tfrecord_file in tfrecord_files:
  if is_tfrecord_corrupted(tfrecord_file):
      print(f"The TFRecord file {tfrecord_file} is corrupted.")
  else:
      print(f"The TFRecord file {tfrecord_file} is fine.")

In [49]:
coco_path = '/local/shared_with_docker/.hailomz/models_files/coco/2021-06-18/coco_val2017.tfrecord'
visdrone_path = '/local/shared_with_docker/.hailomz/models_files/visdrone/2020-05-25/visdrone_val.tfrecord'
raw_dataset = tf.data.TFRecordDataset(coco_path)

for raw_record in raw_dataset.take(1):
    example = tf.train.Example()
    example.ParseFromString(raw_record.numpy())
    print(example)

features {
  feature {
    key: "area"
    value {
      float_list {
        value: 273.6788024902344
        value: 2240.6201171875
        value: 2280.750244140625
        value: 2959.54931640625
        value: 1802.0506591796875
        value: 1329.5894775390625
        value: 4598.166015625
        value: 630.431884765625
        value: 568.4833374023438
        value: 2129.438720703125
        value: 135.81524658203125
        value: 142.76785278320312
        value: 184.18809509277344
        value: 14661.408203125
        value: 531.7581176757812
        value: 278.7001037597656
      }
    }
  }
  feature {
    key: "category_id"
    value {
      int64_list {
        value: 37
        value: 1
        value: 1
        value: 1
        value: 1
        value: 1
        value: 43
        value: 43
        value: 43
        value: 15
        value: 43
        value: 43
        value: 43
        value: 1
        value: 15
        value: 15
      }
    }
  }
  feature {
    key: "

In [ ]:
!ls -la ../data/visdrone/train

In [ ]:
annotation_full_path = '../data/visdrone/val/labels/0000001_02999_d_0000005.txt'
with open(annotation_full_path, "r") as f:
    annotation_file = f.read().splitlines()
annotations = []
for ann_str in annotation_file:
    print(ann_str)
    try:
        line_str = [float(i) for i in ann_str.rstrip(" ").split(" ")]
        category_id, x, y, w, h = (int(line_str[0]), line_str[1], line_str[2], line_str[3], line_str[4])
    except IndexError:
        print(f"problem with {image_filename}")
        raise
    annotations.append([category_id, x, y, w, h])

annotations

### Evaluation

In [289]:
import os
import glob
import numpy as np
from tqdm import tqdm
from collections import defaultdict

# Constants
IOU_THRESHOLDS = np.arange(0.5, 1.0, 0.05)  # mAP@[0.5:0.95]
CONFIDENCE_THRESHOLD = 0.3  # Ignore low-confidence predictions

# Convert YOLO format (normalized) to absolute bbox coordinates
def yolo_to_bbox(yolo_bbox, img_width, img_height):
    class_id, x, y, w, h, conf = yolo_bbox
    x_min = int((x - w / 2) * img_width)
    y_min = int((y - h / 2) * img_height)
    x_max = int((x + w / 2) * img_width)
    y_max = int((y + h / 2) * img_height)
    return class_id, x_min, y_min, x_max, y_max, conf

# Compute IoU
def compute_iou(box1, box2):
    x1, y1, x2, y2 = box1
    x1g, y1g, x2g, y2g = box2

    xi1, yi1 = max(x1, x1g), max(y1, y1g)
    xi2, yi2 = min(x2, x2g), min(y2, y2g)
    
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    box1_area, box2_area = (x2 - x1) * (y2 - y1), (x2g - x1g) * (y2g - y1g)
    
    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0

# Parse YOLO .txt files (GT or Predictions)
def parse_yolo_file(file_path, img_width, img_height, is_prediction=False):
    bboxes = []
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            for line in f.readlines():
                values = list(map(float, line.strip().split()))
                if (is_prediction and len(values) == 6) or (not is_prediction and len(values) == 5):
                    if is_prediction and values[5] < CONFIDENCE_THRESHOLD:
                        continue  # Skip low-confidence detections
                    bboxes.append(yolo_to_bbox(values + ([1.0] if not is_prediction else []), img_width, img_height))
    return bboxes

# Compute per-class AP and mAP
def compute_map(ground_truths, predictions):
    class_ap = defaultdict(list)

    for iou_thresh in IOU_THRESHOLDS:
        class_tp, class_fp, class_fn = defaultdict(int), defaultdict(int), defaultdict(int)

        for img_id, preds in predictions.items():
            gt_bboxes = ground_truths.get(img_id, [])
            matched_gt = set()

            for pred in sorted(preds, key=lambda x: x[-1], reverse=True):  # Sort by confidence
                pred_class, px1, py1, px2, py2, conf = pred
                matched = False

                for gt in gt_bboxes:
                    gt_class, gx1, gy1, gx2, gy2 = gt
                    if pred_class == gt_class and compute_iou((px1, py1, px2, py2), (gx1, gy1, gx2, gy2)) >= iou_thresh:
                        if gt not in matched_gt:
                            class_tp[pred_class] += 1
                            matched_gt.add(gt)
                            matched = True
                            break
                
                if not matched:
                    class_fp[pred_class] += 1

            for gt in gt_bboxes:
                gt_class = gt[0]
                if gt not in matched_gt:
                    class_fn[gt_class] += 1

        for class_id in class_tp.keys():
            tp, fp, fn = class_tp[class_id], class_fp[class_id], class_fn[class_id]
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            class_ap[class_id].append(precision * recall)  # Approximate AP

    # Compute mean AP for each class
    per_class_map = {c: np.mean(ap_list) for c, ap_list in class_ap.items()}
    mean_ap = np.mean(list(per_class_map.values()))

    return mean_ap, per_class_map

# Evaluate model
def evaluate_model(gt_folder, pred_folder, image_size):
    gt_data, pred_data = {}, {}

    for gt_file in tqdm(glob.glob(os.path.join(gt_folder, "*.txt")), desc="Processing GT"):
        file_name = os.path.basename(gt_file)
        gt_data[file_name] = parse_yolo_file(gt_file, *image_size, is_prediction=False)

    for pred_file in tqdm(glob.glob(os.path.join(pred_folder, "*.txt")), desc="Processing Predictions"):
        file_name = os.path.basename(pred_file)
        pred_data[file_name] = parse_yolo_file(pred_file, *image_size, is_prediction=True)
    mean_ap, per_class_map = compute_map(gt_data, pred_data)

    print("\nMean Average Precision (mAP@[0.5:0.95]): {:.4f}".format(mean_ap))
    print("Per-Class AP:")
    for class_id, ap in per_class_map.items():
        print(f"  Class {class_id}: AP = {ap:.4f}")

# Run Evaluation
rootdir = "/local/shared_with_docker/test/"
gtdir = os.path.join(rootdir,"labels/" )
preddir = os.path.join(rootdir,"detections/" )

evaluate_model(gtdir, preddir, image_size=(640, 640))


Processing Predictions: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 6615.62it/s]


ValueError: too many values to unpack (expected 5)

In [304]:
def get_image(path):
    # Load the image
    img = Image.open(SAMPLE_IMAGE_PATH).convert('RGB')
    # Get image dimensions
    width, height = img.size
    
    return img, width, height

def scale_xyxy(xyxy, w, h, imgsz=640):
    # Scaling factors for predictions to map back to original shape
    print("Scaling xyxy detection...\n")
    scale_x = w / imgsz
    scale_y = h / imgsz
    num_dets = xyxy.shape[0]
    preds = np.zeros([num_dets, 4])
    for idx in range(num_dets):
        # Rescale the coordinates
        x1_orig = xyxy[idx, 0] * scale_x
        y1_orig = xyxy[idx, 1] * scale_y
        x2_orig = xyxy[idx, 2] * scale_x
        y2_orig = xyxy[idx, 3] * scale_y
        preds[idx, :] = [x1_orig, y1_orig, x2_orig, y2_orig]
    return preds
    
def generate_dataset(dataset_sz=None, ):
    dataset = np.zeros((data_batch_size, 640, 640, 3))
    for idx, img_name in enumerate(sorted(images_list)):
        if idx==data_batch_size:
            break
        img = Image.open(os.path.join(images_path, img_name)).convert('RGB')
        img_preproc = preproc(img)
        dataset[idx, :, :, :] = img_preproc
    
def get_predictions(runner, img):
    imgproc = np.array(preproc(img)) # basically just reshape it, but can add other transforms
    inputs = imgproc.reshape(1,640,640,3) # need to reshape since batch is expected
    
    with runner.infer_context(InferenceContext.SDK_QUANTIZED) as ctx:
        output = runner.infer(ctx, inputs)
        
    return output

def process_preds(model_outputs):
    detections = output[0]
    outs = []
    for class_idx in range(detections.shape[0]):  # Iterate over the classes
        valid_mask = np.any(detections[class_idx,:,:] != 0, axis=0)  # Check if any non-zero values exist in each column
        last_valid_indices = np.argmax(~valid_mask, axis=0) # First occurrence of zero padding
        if last_valid_indices == 0:
            continue
        dets = output[0, class_idx, :, :last_valid_indices]
        dets = dets.transpose()
        for el in range(last_valid_indices):
            #outs.append(f"{class_idx} {dets[el][0]} {dets[el][1]} {dets[el][2]} {dets[el][3]} {dets[el][4]}")
            outs.append([class_idx, dets[el][0], dets[el][1], dets[el][2], dets[el][3], dets[el][4]])
    return np.array(outs)


def get_detections_xyxy(dets, w, h, imgsz=640):
    """
    Processes the detections from a single image and converts into xyxy format
    For unknown reason, output is in yxyx format, so needs to be flipped
    
    Args:
        x (np.ndarray | torch.Tensor): The bounding box coordinates.
        w (int): Original width of the image.
        h (int): original height of the image.
        imgsz (int): Image size the model expects
        padw (int): Padding width. Defaults to 0
        padh (int): Padding height. Defaults to 0
    Returns:
        xyxy (np.ndarray | torch.Tensor): The coordinates of the bounding box in the format [x1, y1, x2, y2] where
            x1,y1 is the top-left corner, x2,y2 is the bottom-right corner of the bounding box. Relative to the
            original image size, hence rescaled
    """
    xyxy = np.zeros([proc_dets.shape[0], 4])
    for idx, pred in enumerate(proc_dets):
        xyxy[idx, :] = np.array([pred[2], pred[1], pred[4], pred[3]])
    
    xyxy = scale_xyxy(xyxy*640, w=640, h=426, imgsz=640)

    return xyxy


# sample_output =  np.array([[0.39236316, 0.00957916, 0.6129946 , 0.24244972],
#        [0.48750573, 0.8681353 , 0.6807421 , 0.99895185]], 'dtype=float32')

def yolo_to_bbox(yolo_bbox, img_width, img_height):
    class_id, x, y, w, h = yolo_bbox
    x_min = (x - w / 2) * img_width
    y_min = (y - h / 2) * img_height
    x_max = (x + w / 2) * img_width
    y_max = (y + h / 2) * img_height
    
    gtbbox = np.array([class_id, x_min, y_min, x_max, y_max])
    return gtbbox

def compute_iou(det_box, gt_box):
    x1, y1, x2, y2 = det_box
    x1g, y1g, x2g, y2g = gt_box

    xi1, yi1 = max(x1, x1g), max(y1, y1g)
    xi2, yi2 = min(x2, x2g), min(y2, y2g)
    
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    box1_area, box2_area = (x2 - x1) * (y2 - y1), (x2g - x1g) * (y2g - y1g)
    
    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0    

from collections import defaultdict
import numpy as np

def compute_map_single(gt_bbox, pred_bbox, iou_thresholds=[0.5]):
    """
    Compute mean Average Precision (mAP) for a single ground truth and predicted bounding box.
    :param gt_bbox: Tuple (class, x1, y1, x2, y2) representing ground truth bounding box
    :param pred_bbox: Tuple (class, x1, y1, x2, y2, confidence) representing predicted bounding box
    :param iou_thresholds: List of IoU thresholds to evaluate against
    :return: Mean AP and per-class AP dictionary
    """
    class_ap = defaultdict(list)
    pred_class, px1, py1, px2, py2, conf = pred_bbox
    gt_class, gx1, gy1, gx2, gy2 = gt_bbox

    for iou_thresh in iou_thresholds:
        class_tp, class_fp, class_fn = defaultdict(int), defaultdict(int), defaultdict(int)
        
        if pred_class == gt_class and compute_iou((px1, py1, px2, py2), (gx1, gy1, gx2, gy2)) >= iou_thresh:
            class_tp[pred_class] += 1
        else:
            class_fp[pred_class] += 1
            class_fn[gt_class] += 1

        for class_id in set([pred_class, gt_class]):
            tp, fp, fn = class_tp[class_id], class_fp[class_id], class_fn[class_id]
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            print(precision, recall)
            class_ap[class_id].append(precision * recall)  # Approximate AP

    # Compute mean AP for each class
    per_class_map = {c: np.mean(ap_list) for c, ap_list in class_ap.items()}
    mean_ap = np.mean(list(per_class_map.values()))

    return mean_ap, per_class_map




In [305]:
# setup runner
# model_name = "yolo11n"
# quantized_model_har_path = f"{model_name}_quantized_model_opl4.har"
# assert os.path.isfile(quantized_model_har_path), "Please provide valid path for HAR file"
# runner = ClientRunner(har=quantized_model_har_path)

# # sample image
# SAMPLE_IMAGE_PATH = "/local/shared_with_docker/coco139.jpg"

# # get model detections
# img, width, height = get_image(SAMPLE_IMAGE_PATH)
# raw_dets = get_predictions(runner, img)
# proc_dets = process_preds(raw_dets)
#dets = get_detections_xyxy(proc_dets, w=width, h=height, imgsz=640)

# convert gt xywh to xyxy
gtlabels = [62, 0.127641, 0.505153, 0.233312, 0.2227]
gtxyxy  = yolo_to_bbox(gtlabels, 640, 426)

compute_iou(dets[7], gtxyxy[1:])

compute_map_single(gtxyxy, np.array([62, *dets[7], 0.95]))





1.0 1.0


(1.0, {62.0: 1.0})

In [36]:
def xywhn2xyxy(x, w=640, h=640, padw=0, padh=0):
    """
    Convert normalized bounding box coordinates to pixel coordinates.

    Args:
        x (np.ndarray | torch.Tensor): The bounding box coordinates.
        w (int): Width of the image. Defaults to 640
        h (int): Height of the image. Defaults to 640
        padw (int): Padding width. Defaults to 0
        padh (int): Padding height. Defaults to 0
    Returns:
        y (np.ndarray | torch.Tensor): The coordinates of the bounding box in the format [x1, y1, x2, y2] where
            x1,y1 is the top-left corner, x2,y2 is the bottom-right corner of the bounding box.
    """
    assert x.shape[-1] == 4, f"input shape last dimension expected 4 but input shape is {x.shape}"
    y = np.empty_like(x)  # faster than clone/copy
    y[..., 0] = w * (x[..., 0] - x[..., 2] / 2) + padw  # top left x
    y[..., 1] = h * (x[..., 1] - x[..., 3] / 2) + padh  # top left y
    y[..., 2] = w * (x[..., 0] + x[..., 2] / 2) + padw  # bottom right x
    y[..., 3] = h * (x[..., 1] + x[..., 3] / 2) + padh  # bottom right y
    return y

xywh = np.array([0.127641, 0.505153, 0.233312, 0.2227])

xywhn2xyxy(xywh, w=640, h=426)

array([  7.0304  , 167.760078, 156.35008 , 262.630278])

In [268]:
detections = output[0]  # Shape (80, 5, 100)
num_detections = 0
outs = []

with open('detections.txt', 'w') as file:
    for class_idx in range(detections.shape[0]):  # Iterate over the classes
        valid_mask = np.any(detections[class_idx,:,:] != 0, axis=0)  # Check if any non-zero values exist in each column
        last_valid_indices = np.argmax(~valid_mask, axis=0) # First occurrence of zero padding
        if last_valid_indices == 0:
            continue
        dets = output[0, class_idx, :, :last_valid_indices]
        dets = dets.transpose()
        for el in range(last_valid_indices):
            outs.append(f"{class_idx} {dets[el][0]} {dets[el][1]} {dets[el][2]} {dets[el][3]} {dets[el][4]}\n")

    file.write(''.join(map(str, outs)))
    file.close()


In [139]:
gtlabels = '/local/workspace/hailo_virtualenv/lib/python3.10/site-packages/hailo_tutorials/data/coco/test/labels/'

test = output[0,62, :, :]
print(test.shape)
# remove all non-zero padding elements
valid_mask = np.any(output != 0, axis=(1, 2))  # Check if any non-zero values exist in each column
last_valid_indices = np.argmax(~valid_mask, axis=1) # First occurrence of zero padding
print(last_valid_indices)
last_valid_indices[valid_mask[:, -1]] = output.shape[-1] # edge case, no detections
dets = output[0, 62, :, :last_valid_indices[0]-1]

dets = dets.transpose()
dets

(5, 100)
[3]


array([[0.39342952, 0.00958287, 0.615698  , 0.2424439 , 0.8860524 ],
       [0.48757473, 0.8674938 , 0.68027043, 0.9978351 , 0.53390336]],
      dtype=float32)

In [375]:
import json
import random


# COCO to YOLO category mapping
coco_to_yolo = {
    1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6, 8: 7, 9: 8, 10: 9, 11: 10, 13: 11, 14: 12,
    15: 13, 16: 14, 17: 15, 18: 16, 19: 17, 20: 18, 21: 19, 22: 20, 23: 21, 24: 22, 25: 23,
    27: 24, 28: 25, 31: 26, 32: 27, 33: 28, 34: 29, 35: 30, 36: 31, 37: 32, 38: 33, 39: 34,
    40: 35, 41: 36, 42: 37, 43: 38, 44: 39, 46: 40, 47: 41, 48: 42, 49: 43, 50: 44, 51: 45,
    52: 46, 53: 47, 54: 48, 55: 49, 56: 50, 57: 51, 58: 52, 59: 53, 60: 54, 61: 55, 62: 56,
    63: 57, 64: 58, 65: 59, 67: 60, 70: 61, 72: 62, 73: 63, 74: 64, 75: 65, 76: 66, 77: 67,
    78: 68, 79: 69, 80: 70, 81: 71, 82: 72, 84: 73, 85: 74, 86: 75, 87: 76, 88: 77, 89: 78, 90: 79
}


def create_results_file(inputfile, outputfile, isCoco=False, imgId=None, catId=None):
    # Load the COCO 2014 JSON file
    with open(inputfile, "r") as file:  # Change to your file path
        coco_data = json.load(file)


        
    # Extract and map annotations
    annotations = []
    annId = 1
    for ann in coco_data["annotations"]:
        if ann["image_id"] in imgId and ann["category_id"] == catId:
            coco_category_id = ann["category_id"]
            if isCoco and coco_category_id in coco_to_yolo:
                mapped_category_id = coco_category_id[coco_toyolo] # need to validate this
                annotations.append({
                    "image_id": ann["image_id"],
                    "category_id": mapped_category_id,  # Mapped category ID
                    "bbox": ann["bbox"],  # bbox in xywh format
                    "score": ann.get("score", None)  # Some files may not have a score
                })
            else:
                annotations.append({
                    "image_id": ann["image_id"],
                    "category_id": coco_category_id,  # Mapped category ID
                    "bbox": ann["bbox"],  # bbox in xywh format
                    "score": round(random.uniform(0.7, 0.99), 5),
                    "id": annId,
                    #"segmentation": []
                })
        annId += 1
    print(f"{len(annotations)} annotations extracted, writing to output file")
    with open(outputfile, 'w') as outf:
        json.dump(annotations, outf)

infile = "/local/shared_with_docker/visdrone/annotations_VisDrone_val.json"
outfile = "/local/shared_with_docker/visdrone/results.json"
imglist = [2]
create_results_file(infile, outfile, False, imglist, 1)     

9 annotations extracted, writing to output file


### Create a sample annotations file with just a subset
Not really needed since can just use imgId param in cocoeval object to filter images

In [363]:
import json

# Load the COCO JSON file (update the filename if needed)
input_filename = "/local/shared_with_docker/coco_eval/annotations/instances_val2017.json"
output_filename = "/local/shared_with_docker/coco_eval/coco/annotations/instnaces_train2017_img0.json"

with open(input_filename, "r") as file:
    coco_data = json.load(file)

# Image ID to keep
image_id_to_keep = 1
#cat_id_to_keep = 5

# Filter only annotations for image ID 139
filtered_annotations = [ann for ann in coco_data["annotations"] if ann["image_id"] == image_id_to_keep]
filtered_annotations = [ann for ann in filtered_annotations if ann["category_id"] == cat_id_to_keep]

filtered_images = [img for img in coco_data["images"] if img["id"] == image_id_to_keep]

# Replace annotations in the dataset
coco_data["annotations"] = filtered_annotations
coco_data["images"] = filtered_images

# Save the modified JSON file
with open(output_filename, "w") as outfile:
    json.dump(coco_data, outfile)

print(f"Filtered JSON saved to {output_filename} with only annotations for image ID {image_id_to_keep}")


Filtered JSON saved to /local/shared_with_docker/coco_eval/coco/annotations/instnaces_train2017_img0.json with only annotations for image ID 1
